# Robotics Core Math Demos

This notebook demonstrates core mathematical concepts in robotics:
- Position description
- Rotation matrices about coordinate axes
- Euler angles and gimbal lock
- Quaternions
- Homogeneous transformation matrices
- Transform chains

Uses `%matplotlib widget` for interactive 3D visualization

In [ ]:
# Set interactive backend (prefer widget, fallback to inline)
from IPython import get_ipython
from IPython.display import HTML, display
import os

ip = get_ipython()
is_vscode = (
    bool(os.environ.get('VSCODE_PID'))
    or bool(os.environ.get('VSCODE_CWD'))
    or 'VSCODE' in os.environ.get('TERM_PROGRAM', '').upper()
)

if ip is not None:
    try:
        ip.run_line_magic('matplotlib', 'widget')  # requires ipympl
        backend_mode = 'widget'
    except Exception:
        ip.run_line_magic('matplotlib', 'inline')
        backend_mode = 'inline'
else:
    backend_mode = 'script'

# Import all dependencies
import numpy as np
from spatialmath import SE3, SO3, UnitQuaternion
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from matplotlib import font_manager

# Register Times New Roman font with LaTeX rendering
font_manager.fontManager.addfont("/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf")
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["text.usetex"] = True
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}\usepackage{bm}"

# Enable animation display in inline mode
plt.rcParams['animation.html'] = 'jshtml'
plt.rcParams['animation.embed_limit'] = 100  # MB


def show_animation(anim, fig=None):
    """Robust animation display: prefer jshtml, fallback to plt.show()."""
    try:
        display(HTML(anim.to_jshtml()))
        if fig is not None:
            plt.close(fig)
    except Exception as e:
        print(f"Warning: jshtml display failed: {e}")
        try:
            plt.show()
        except Exception as show_err:
            print(f"Warning: plt.show() also failed: {show_err}")


print(f"Setup complete (matplotlib mode: {backend_mode}, vscode={is_vscode})")

## Helper Functions

In [ ]:
# Animation constants
ANIMATION_FRAMES = 60
ANIMATION_INTERVAL = 50  # milliseconds


def plot_frame(ax, T, label='', length=0.3, linewidth=2):
    """
    Plot a 3D coordinate frame.

    Args:
        ax: matplotlib 3D axes
        T: SE3 transformation matrix
        label: frame label
        length: axis length
        linewidth: line width
    """
    # Input validation
    if not hasattr(ax, 'get_zlim'):
        raise ValueError("ax must be a 3D axes object")
    if not isinstance(T, SE3):
        raise TypeError("T must be an SE3 object")

    origin = T.t
    x_axis = T.R @ np.array([length, 0, 0])
    y_axis = T.R @ np.array([0, length, 0])
    z_axis = T.R @ np.array([0, 0, length])

    # Draw three coordinate axes
    ax.quiver(origin[0], origin[1], origin[2],
              x_axis[0], x_axis[1], x_axis[2],
              color='r', arrow_length_ratio=0.2, linewidth=linewidth)
    ax.quiver(origin[0], origin[1], origin[2],
              y_axis[0], y_axis[1], y_axis[2],
              color='g', arrow_length_ratio=0.2, linewidth=linewidth)
    ax.quiver(origin[0], origin[1], origin[2],
              z_axis[0], z_axis[1], z_axis[2],
              color='b', arrow_length_ratio=0.2, linewidth=linewidth)

    if label:
        ax.text(origin[0], origin[1], origin[2], label, fontsize=10, weight='bold')


def setup_3d_axes(ax, xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1), title=''):
    """
    Set up 3D axes.

    Args:
        ax: matplotlib 3D axes
        xlim, ylim, zlim: axis limits
        title: plot title
    """
    # Input validation
    if not hasattr(ax, 'get_zlim'):
        raise ValueError("ax must be a 3D axes object")

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)
    ax.set_xlabel(r'$X$', fontsize=10)
    ax.set_ylabel(r'$Y$', fontsize=10)
    ax.set_zlabel(r'$Z$', fontsize=10)
    if title:
        ax.set_title(title, fontsize=12, weight='bold')
    ax.grid(True, alpha=0.3)

## Demo 1: Position Description

Demonstrates the representation of point P in different coordinate frames and coordinate transformations.

In [ ]:
def demo1_position_description():
    """
    Demo 1: Position Description - Coordinate transform via moving frame (Z always up)
    """
    print("\n=== Demo 1: Position Description ===")

    fig = plt.figure(figsize=(14, 8))
    ax = fig.add_subplot(111, projection='3d')

    A_T_A = SE3()
    gamma_target = np.pi / 4
    A_t_AB_target = np.array([0.5, 0.3, 0.2])
    B_p_h = np.array([0.3, 0.2, 0.1, 1.0])
    print(f"B_p = {B_p_h[:3]}")

    A_T_B_list = []
    A_p_list = []
    A_o_B_list = []

    for i in range(ANIMATION_FRAMES + 1):
        tau = i / ANIMATION_FRAMES
        gamma = gamma_target * tau
        A_t_AB = A_t_AB_target * tau

        A_T_B = SE3.Rz(gamma) * SE3.Trans(*A_t_AB)
        A_p_h = A_T_B.A @ B_p_h

        A_T_B_list.append(A_T_B)
        A_p_list.append(A_p_h[:3])
        A_o_B_list.append(A_T_B.t)

    A_p_list = np.array(A_p_list)
    A_o_B_list = np.array(A_o_B_list)

    def update(frame):
        ax.clear()

        A_T_B = A_T_B_list[frame]
        A_p_h = A_T_B.A @ B_p_h
        A_p = A_p_h[:3]
        A_o_B = A_T_B.t

        plot_frame(ax, A_T_A, r'$\{A\}$', 0.4, linewidth=1)
        plot_frame(ax, A_T_B, r'$\{B\}$', 0.4, linewidth=2)

        ax.scatter(*A_o_B, color='orange', s=70, label=r'$\{B\}$ origin')
        ax.scatter(*A_p, color='red', s=100, label=r'Point $P$')

        if frame > 0:
            origin_slice = A_o_B_list[:frame + 1]
            point_slice = A_p_list[:frame + 1]
            ax.plot(origin_slice[:, 0], origin_slice[:, 1], origin_slice[:, 2],
                    'k:', alpha=0.7, linewidth=1, label=r'Origin trajectory')
            ax.plot(point_slice[:, 0], point_slice[:, 1], point_slice[:, 2],
                    'r--', alpha=0.6, linewidth=1, label=r'Point trajectory')

        setup_3d_axes(
            ax,
            xlim=(-0.2, 1.2),
            ylim=(-0.2, 1.0),
            zlim=(-0.2, 0.6),
            title=r'Demo 1: Position Description'
        )
        ax.legend(loc='upper right', fontsize=8)

        R = A_T_B.R
        t = A_T_B.t

        formula_text = (
            r'${}^{A}\mathbf{p} = {}^{A}\mathbf{T}_{B} \cdot {}^{B}\mathbf{p}$'
            '\n\n'
            r'${}^{A}\mathbf{T}_{B} = \begin{bmatrix}'
            + f'{R[0,0]:+.3f} & {R[0,1]:+.3f} & {R[0,2]:+.3f} & {t[0]:+.3f}' + r' \\'
            + f' {R[1,0]:+.3f} & {R[1,1]:+.3f} & {R[1,2]:+.3f} & {t[1]:+.3f}' + r' \\'
            + f' {R[2,0]:+.3f} & {R[2,1]:+.3f} & {R[2,2]:+.3f} & {t[2]:+.3f}' + r' \\'
            + r' 0 & 0 & 0 & 1'
            + r'\end{bmatrix}$'
            '\n\n'
            r'${}^{B}\mathbf{p} = \begin{bmatrix}'
            + f'{B_p_h[0]:.1f}' + r' \\ '
            + f'{B_p_h[1]:.1f}' + r' \\ '
            + f'{B_p_h[2]:.1f}' + r' \\ '
            + f'{B_p_h[3]:.1f}'
            + r'\end{bmatrix}$'
            r'$\;\;\Rightarrow\;\;{}^{A}\mathbf{p} = \begin{bmatrix}'
            + f'{A_p_h[0]:+.3f}' + r' \\ '
            + f'{A_p_h[1]:+.3f}' + r' \\ '
            + f'{A_p_h[2]:+.3f}' + r' \\ '
            + f'{A_p_h[3]:+.3f}'
            + r'\end{bmatrix}$'
        )

        ax.text2D(
            0.02,
            0.98,
            formula_text,
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85)
        )

    anim = FuncAnimation(
        fig,
        update,
        frames=ANIMATION_FRAMES,
        interval=ANIMATION_INTERVAL,
        repeat=True
    )

    show_animation(anim, fig)
    return fig, anim

# Run demo
demo1_position_description()

## Demo 2: Rotation Matrices about Coordinate Axes

Demonstrates the basic rotation matrices about the x, y, z axes.

In [ ]:
def demo2_axis_rotation_matrices():
    """
    Demo 2: Rotation Matrices about Coordinate Axes
    """
    print("\n=== Demo 2: Rotation Matrices about Coordinate Axes ===")

    frames = 90
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    rotations_x = []
    rotations_y = []
    rotations_z = []

    for i in range(frames + 1):
        tau = i / frames
        theta = np.pi * tau
        R_x = SO3.Rx(theta)
        R_y = SO3.Ry(theta)
        R_z = SO3.Rz(theta)
        rotations_x.append(R_x)
        rotations_y.append(R_y)
        rotations_z.append(R_z)

    def update(frame):
        ax.clear()
        tau = frame / frames
        theta = np.pi * tau

        R_x = rotations_x[frame]
        R_y = rotations_y[frame]
        R_z = rotations_z[frame]

        T_Rx = SE3.Rt(R_x.R, [1, 0, 0])
        T_Ry = SE3.Rt(R_y.R, [0, 1, 0])
        T_Rz = SE3.Rt(R_z.R, [0, 0, 1])

        plot_frame(ax, SE3(), r'$\{A\}$', 0.3)
        plot_frame(ax, T_Rx, r'$R_x(\theta)$', 0.3)
        plot_frame(ax, T_Ry, r'$R_y(\theta)$', 0.3)
        plot_frame(ax, T_Rz, r'$R_z(\theta)$', 0.3)

        setup_3d_axes(
            ax,
            xlim=(-1.5, 1.5),
            ylim=(-1.5, 1.5),
            zlim=(-0.5, 1.5),
            title=r'Demo 2: Axis Rotation Matrices ($\theta=' + f'{theta:.2f}' + r'$ rad)'
        )
        ax.text2D(
            0.05,
            0.95,
            r'$R_x(\theta)$: rotation about $x$-axis' + '\n'
            r'$R_y(\theta)$: rotation about $y$-axis' + '\n'
            r'$R_z(\theta)$: rotation about $z$-axis',
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment='top'
        )

    anim = FuncAnimation(fig, update, frames=frames,
                         interval=ANIMATION_INTERVAL, repeat=True)

    show_animation(anim, fig)
    return fig, anim

# Run demo
demo2_axis_rotation_matrices()

## Demo 3: Euler Angles and Gimbal Lock

Demonstrates the singularity of Euler angles when pitch approaches +/-90 degrees.

Key observations:
- Uses ZYX / RPY angle convention
- When pitch approaches +/-90 degrees, the yaw and roll axes become aligned
- The orientation itself still exists, but the Euler angle parameterization loses one degree of freedom

In [ ]:
def demo3_euler_gimbal_lock():
    """
    Demo 3: Euler Angles and Gimbal Lock
    """
    print("\n=== Demo 3: Euler Angles and Gimbal Lock ===")

    frames = 120
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    rotations = []
    pitch_thetas = []
    roll_phis = []
    yaw_psis = []
    gimbal_indicators = []

    for i in range(frames + 1):
        tau = i / frames
        roll_phi = np.pi / 4 * tau
        pitch_theta = np.pi / 2 * 0.99 * tau
        yaw_psi = np.pi / 3 * tau

        R = SO3.RPY([roll_phi, pitch_theta, yaw_psi])
        gimbal_indicator = abs(np.sin(pitch_theta))

        rotations.append(R)
        pitch_thetas.append(pitch_theta)
        roll_phis.append(roll_phi)
        yaw_psis.append(yaw_psi)
        gimbal_indicators.append(gimbal_indicator)

    def update(frame):
        ax.clear()

        R = rotations[frame]
        pitch_theta = pitch_thetas[frame]
        roll_phi = roll_phis[frame]
        yaw_psi = yaw_psis[frame]
        gimbal_indicator = gimbal_indicators[frame]

        A_T_B = SE3.Rt(R.R, [0, 0, 0])

        plot_frame(ax, SE3(), r'$\{A\}$', 0.4, linewidth=1)
        plot_frame(ax, A_T_B, r'$\{B\}$', 0.4, linewidth=2)

        setup_3d_axes(ax, xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1),
                      title=r'Demo 3: Euler Angles \& Gimbal Lock (ZYX / RPY)')

        warning_text = (
            r'ZYX Euler / RPY' + '\n'
            r'$\phi = ' + f'{roll_phi:.2f}' + r'\;\mathrm{rad}\;(' + f'{np.degrees(roll_phi):.1f}' + r'^{\circ})$  (roll)' + '\n'
            r'$\theta = ' + f'{pitch_theta:.2f}' + r'\;\mathrm{rad}\;(' + f'{np.degrees(pitch_theta):.1f}' + r'^{\circ})$  (pitch)' + '\n'
            r'$\psi = ' + f'{yaw_psi:.2f}' + r'\;\mathrm{rad}\;(' + f'{np.degrees(yaw_psi):.1f}' + r'^{\circ})$  (yaw)' + '\n\n'
            r'As $\theta \to \pm 90^{\circ}$, yaw and roll axes align.' + '\n'
            r'Euler parameterization loses 1 DOF.' + '\n'
            r'$|\sin\theta| = ' + f'{gimbal_indicator:.3f}' + r'$'
        )

        bbox_color = 'mistyrose' if gimbal_indicator > 0.95 else 'wheat'
        alpha = 0.85 if gimbal_indicator > 0.95 else 0.65

        ax.text2D(
            0.03, 0.97, warning_text,
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor=bbox_color, alpha=alpha),
        )

        explanation_text = (
            r'Explanation:' + '\n'
            r'1. Outer yaw ring and inner roll ring' + '\n'
            r'   approach parallel alignment.' + '\n'
            r'2. Small yaw/roll changes produce similar' + '\n'
            r'   orientation changes.' + '\n'
            r'3. Singularity is in the parameterization,' + '\n'
            r'   not the physical orientation.'
        )

        ax.text2D(
            0.03, 0.28, explanation_text,
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85)
        )

    anim = FuncAnimation(fig, update, frames=frames,
                         interval=ANIMATION_INTERVAL, repeat=True)

    show_animation(anim, fig)
    return fig, anim

# Run demo
demo3_euler_gimbal_lock()

## Demo 4: Quaternions

Demonstrates quaternion representation of rotations and Spherical Linear Interpolation (SLERP).

Key observations:
- Quaternion notation: q = [w, x, y, z]
- q = cos(theta/2) + k sin(theta/2), where k is the unit rotation axis
- Unit quaternions avoid the singular parameterization of Euler angles near gimbal lock

In [ ]:
def demo4_quaternion_basics():
    """
    Demo 4: Quaternions & SLERP
    """
    print("\n=== Demo 4: Quaternions ===")

    frames = 90
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Target quaternion: compose Rx(pi/4) * Ry(pi/3)
    q_target = UnitQuaternion.Rx(np.pi / 4) * UnitQuaternion.Ry(np.pi / 3)
    q_identity = UnitQuaternion()

    # Precompute
    quaternions = []
    axis_angles = []

    for i in range(frames + 1):
        tau = i / frames
        q_tau = q_identity.interp(q_target, tau)
        quaternions.append(q_tau)
        theta, k = q_tau.angvec()
        axis_angles.append((k, theta))

    def update(frame):
        ax.clear()
        tau = frame / frames

        q_tau = quaternions[frame]
        w = q_tau.s
        x, y, z = q_tau.v
        k, theta = axis_angles[frame]

        A_T_B = SE3.Rt(q_tau.R, [0, 0, 0])

        plot_frame(ax, SE3(), r'$\{A\}$', 0.4, linewidth=1)
        plot_frame(ax, A_T_B, r'$\{B\}$', 0.4, linewidth=2)

        # Draw rotation axis k
        if theta > 1e-6:
            ax.quiver(0, 0, 0, k[0]*0.6, k[1]*0.6, k[2]*0.6,
                      color='purple', linewidth=2, arrow_length_ratio=0.15,
                      linestyle='--', alpha=0.7)
            ax.text(k[0]*0.65, k[1]*0.65, k[2]*0.65, r'$\bm{k}$', fontsize=12,
                    color='purple')

        setup_3d_axes(ax, xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1),
                      title=r'Demo 4: Quaternions \& SLERP')

        # Annotation box with LaTeX
        half_theta = theta / 2
        cos_ht = np.cos(half_theta)
        sin_ht = np.sin(half_theta)

        defn_text = (
            r'$\mathbf{q} = [w,\; x,\; y,\; z] = \cos\!\dfrac{\theta}{2} + \bm{k}\,\sin\!\dfrac{\theta}{2}$'
            '\n\n'
            r'$\bm{k} = [' + f'{k[0]:+.3f},\\; {k[1]:+.3f},\\; {k[2]:+.3f}' + r']$' + '\n'
            r'$\theta = ' + f'{theta:.3f}' + r'\;\mathrm{rad}\;(' + f'{np.degrees(theta):.1f}' + r'^{\circ})$'
            '\n\n'
            r'$\cos(\theta/2) = \cos(' + f'{half_theta:.3f}' + r') = ' + f'{cos_ht:.4f}' + r'$' + '\n'
            r'$\sin(\theta/2) = \sin(' + f'{half_theta:.3f}' + r') = ' + f'{sin_ht:.4f}' + r'$' + '\n'
            r'$\bm{k}\sin(\theta/2) = [' + f'{k[0]*sin_ht:+.4f},\\; {k[1]*sin_ht:+.4f},\\; {k[2]*sin_ht:+.4f}' + r']$'
            '\n\n'
            r'$\Rightarrow\;\mathbf{q} = [' + f'{w:+.4f},\\; {x:+.4f},\\; {y:+.4f},\\; {z:+.4f}' + r']$' + '\n'
            r'$\|\mathbf{q}\| = ' + f'{np.sqrt(w**2+x**2+y**2+z**2):.4f}' + r'$'
        )

        ax.text2D(
            0.02, 0.98, defn_text,
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.85),
        )

    anim = FuncAnimation(fig, update, frames=frames,
                         interval=ANIMATION_INTERVAL, repeat=True)

    show_animation(anim, fig)
    return fig, anim

# Run demo
demo4_quaternion_basics()

## Demo 5: Homogeneous Transformation Matrix

Demonstrates how homogeneous transformation matrices unify rotation and translation.

In [ ]:
def demo5_homogeneous_transform():
    """
    Demo 5: Homogeneous Transformation Matrix
    """
    print("\n=== Demo 5: Homogeneous Transformation Matrix ===")

    frames = 90
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    A_T_B_list = []
    A_t_AB_list = []

    for i in range(frames + 1):
        tau = i / frames
        theta = np.pi / 2 * tau
        A_R_B = SO3.Rz(theta)
        A_t_AB = np.array([0.5 * tau, 0.3 * tau, 0.2 * tau])
        A_T_B = SE3.Rt(A_R_B.R, A_t_AB)
        A_T_B_list.append(A_T_B)
        A_t_AB_list.append(A_t_AB)

    A_t_AB_list = np.array(A_t_AB_list)

    def update(frame):
        ax.clear()
        tau = frame / frames
        theta = np.pi / 2 * tau

        A_T_B = A_T_B_list[frame]

        plot_frame(ax, SE3(), r'$\{A\}$', 0.3, linewidth=1)
        plot_frame(ax, A_T_B, r'$\{B\}$', 0.3, linewidth=2)

        if frame > 0:
            traj_slice = A_t_AB_list[:frame + 1]
            ax.plot(traj_slice[:, 0], traj_slice[:, 1], traj_slice[:, 2],
                    'b-', linewidth=2, label=r'$\mathbf{t}$ trajectory')

        setup_3d_axes(ax, xlim=(-0.2, 0.7), ylim=(-0.2, 0.5), zlim=(-0.2, 0.4),
                      title=r'Demo 5: Homogeneous Transformation Matrix')

        R = A_T_B.R
        t = A_T_B.t
        info_text = (
            r'$\theta = ' + f'{theta:.2f}' + r'\;\mathrm{rad}$' + '\n\n'
            r'${}^{A}\mathbf{t}_{B} = [' + f'{t[0]:.3f},\\; {t[1]:.3f},\\; {t[2]:.3f}' + r']^T$' + '\n\n'
            r'${}^{A}\mathbf{T}_{B} = \begin{bmatrix}'
            + f'{R[0,0]:.3f} & {R[0,1]:.3f} & {R[0,2]:.3f} & {t[0]:.3f}' + r' \\'
            + f' {R[1,0]:.3f} & {R[1,1]:.3f} & {R[1,2]:.3f} & {t[1]:.3f}' + r' \\'
            + f' {R[2,0]:.3f} & {R[2,1]:.3f} & {R[2,2]:.3f} & {t[2]:.3f}' + r' \\'
            + r' 0 & 0 & 0 & 1'
            + r'\end{bmatrix}$'
        )

        ax.text2D(
            0.05, 0.95, info_text,
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7),
        )
        if frame > 0:
            ax.legend(loc='upper right')

    anim = FuncAnimation(fig, update, frames=frames,
                         interval=ANIMATION_INTERVAL, repeat=True)

    show_animation(anim, fig)
    return fig, anim

# Run demo
demo5_homogeneous_transform()

## Demo 6: Transform Chain

Demonstrates the composition of multiple coordinate transformations.

In [ ]:
def demo6_transform_chain():
    """
    Demo 6: Transform Chain
    """
    print("\n=== Demo 6: Transform Chain ===")

    frames = 120
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    transform_chains = []

    for i in range(frames + 1):
        tau = i / frames

        A_T_B = SE3.Rz(np.pi / 2 * tau) * SE3.Tx(0.3)
        B_T_C = SE3.Rz(np.pi * tau) * SE3.Tx(0.3)
        C_T_D = SE3.Rz(1.5 * np.pi * tau) * SE3.Tx(0.2)

        A_T_C = A_T_B * B_T_C
        A_T_D = A_T_C * C_T_D

        transform_chains.append((A_T_B, A_T_C, A_T_D))

    def update(frame):
        ax.clear()

        A_T_B, A_T_C, A_T_D = transform_chains[frame]
        A_T_A = SE3()

        plot_frame(ax, A_T_A, r'$\{A\}$', 0.2, linewidth=1)
        plot_frame(ax, A_T_B, r'$\{B\}$', 0.2, linewidth=1)
        plot_frame(ax, A_T_C, r'$\{C\}$', 0.2, linewidth=1)
        plot_frame(ax, A_T_D, r'$\{D\}$', 0.2, linewidth=2)

        origins = [A_T_A.t, A_T_B.t, A_T_C.t, A_T_D.t]
        for i in range(len(origins) - 1):
            ax.plot([origins[i][0], origins[i + 1][0]],
                    [origins[i][1], origins[i + 1][1]],
                    [origins[i][2], origins[i + 1][2]],
                    'k-', linewidth=3, alpha=0.6)

        for origin in origins:
            ax.scatter(*origin, color='red', s=100, zorder=5)

        setup_3d_axes(ax, xlim=(-1, 1), ylim=(-1, 1), zlim=(-0.5, 1),
                      title=r'Demo 6: Transform Chain')

        ax.text2D(
            0.05,
            0.95,
            r'${}^{A}\mathbf{T}_{D} = {}^{A}\mathbf{T}_{B} \cdot {}^{B}\mathbf{T}_{C} \cdot {}^{C}\mathbf{T}_{D}$'
            + '\n' +
            r'Multi-frame transforms compose sequentially',
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5)
        )

    anim = FuncAnimation(fig, update, frames=frames,
                         interval=ANIMATION_INTERVAL, repeat=True)

    show_animation(anim, fig)
    return fig, anim

# Run demo
demo6_transform_chain()